# Vector Memory Agent — Semantic Memory Retrieval with ChromaDB

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Esturban/agent/blob/master/examples/87-vector-memory-agent/vector_memory_agent_workbook.ipynb)

Contrast with Redis (example 82): **Redis = full ordered history** (grows with every turn); **ChromaDB = top-k semantically relevant turns** (constant context size). Vector memory scales to thousands of turns without growing the context window.

---

### Workshop Roadmap

| # | Section |
|---|----------|
| 1 | **Why Vector Memory** — Redis vs ChromaDB trade-offs |
| 2 | **Storing and Retrieving** — embed, add, query |
| 3 | **Semantic Retrieval** — cosine similarity in action |
| 4 | **LangGraph Workflow** — retrieve → respond → store |
| 5 | **Scaling Advantage** — O(k) context regardless of history size |
| ★ | **Exercises** — top_k tuning, user isolation |

In [ ]:
import sys

def _in_colab():
    try:
        import google.colab; return True
    except ImportError:
        return False

if _in_colab():
    import subprocess
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
        'chromadb', 'langchain-openai', 'langgraph', 'python-dotenv'])
    print('Colab install complete.')
else:
    print('Local environment — skipping install.')

In [ ]:
import os

# Colab stores secrets in userdata; local dev uses a .env file
try:
    from google.colab import userdata
    os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
except ImportError:
    from dotenv import load_dotenv; load_dotenv()

key = os.environ.get('OPENAI_API_KEY', '')
if not key:
    raise RuntimeError('OPENAI_API_KEY is required for the live vector-memory cells.')
print('API key loaded.')

## Part 1 — Why Vector Memory?

**The problem**: a full conversation history of 1 000 turns costs ~100 K tokens per request. Most of it is irrelevant to the current query.

**Vector memory solution**: embed each turn, store it in ChromaDB, retrieve only the top-*k* most semantically similar turns at query time.

**Key concept — cosine similarity**: turns that discuss the same topic have similar embeddings and score high; unrelated turns score low and are never retrieved.

| | Redis (example 82) | ChromaDB (this example) |
|---|---|---|
| Order | Ordered (newest first) | Unordered (by similarity) |
| Recall | Full history | Top-k relevant |
| Context cost | O(N) — grows forever | O(k) — constant |
| Best for | Sequential dialogue | Long-term personal memory |

In [ ]:
import uuid
import chromadb
from langchain_openai import OpenAIEmbeddings

# In-memory ChromaDB client — no persistence, fresh each run
# Use chromadb.PersistentClient('./chroma_data') for production persistence
chroma_client = chromadb.Client()

# text-embedding-3-small: 1536 dims, fast, cheap — good for memory retrieval
embeddings = OpenAIEmbeddings(model='text-embedding-3-small')

# Get or create a collection with cosine similarity
collection = chroma_client.get_or_create_collection(
    name='conversation_memory',
    metadata={'hnsw:space': 'cosine'},   # cosine similarity for text
)

print(f'Collection created: {collection.name}')
print(f'Documents in collection: {collection.count()}')

## Part 2 — Storing and Retrieving Turns

**Storing**: `collection.add(ids, embeddings, documents, metadatas)` — embed the text and store it with metadata.

**Retrieving**: `collection.query(query_embeddings, n_results, where)` — embed the query and find the most similar stored texts.

**Per-user isolation**: `where={"user_id": user_id}` filters results to a specific user's turns. Each user's memory is invisible to other users.

Each turn gets a **UUID** as its ID — guarantees uniqueness even for identical text stored multiple times.

In [ ]:
USER_ID = 'learner-87'

def store_turn(text: str, role: str) -> str:
    '''Embed and store one conversation turn.'''
    turn_id = str(uuid.uuid4())
    embedding = embeddings.embed_query(text)
    collection.add(
        ids=[turn_id],
        embeddings=[embedding],
        documents=[text],
        metadatas=[{'user_id': USER_ID, 'role': role}],
    )
    return turn_id

def retrieve_relevant(query: str, top_k: int = 3) -> list[str]:
    '''Retrieve the most semantically similar turns.'''
    user_turns = collection.get(where={'user_id': USER_ID}, include=[])['ids']
    if not user_turns:
        return []
    query_embedding = embeddings.embed_query(query)
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=min(top_k, len(user_turns)),
        where={'user_id': USER_ID},
    )
    return results['documents'][0] if results['documents'] else []

# Seed some conversation history
seed_turns = [
    ('My name is Jordan and I\'m a data scientist.', 'user'),
    ('Nice to meet you, Jordan! Data science is a great field.', 'assistant'),
    ('I mainly use Python and SQL at work.', 'user'),
    ('Python and SQL are an excellent combination for data work.', 'assistant'),
    ('My favourite ML framework is PyTorch.', 'user'),
    ('PyTorch has great flexibility for research and production.', 'assistant'),
    ('I\'m currently learning LangGraph for agent workflows.', 'user'),
    ('LangGraph is excellent for building stateful agents!', 'assistant'),
]

for text, role in seed_turns:
    store_turn(text, role)

print(f'Stored {collection.count()} turns in ChromaDB.')

## Part 3 — Semantic Retrieval in Action

Instead of loading the full history, we embed the query and find the most **topically similar** past turns. Notice: the retrieval finds *relevant* turns, not necessarily *recent* ones.

This is the key difference from Redis: Redis returns the last N turns regardless of topic; ChromaDB returns the top-k turns closest in meaning to the query.

In [ ]:
test_queries = [
    'What\'s my name?',
    'What tools do I use at work?',
    'What am I learning right now?',
]

for query in test_queries:
    relevant = retrieve_relevant(query, top_k=2)
    print(f'Query: {query}')
    print(f'Retrieved ({len(relevant)} turns):')
    for turn in relevant:
        print(f'  - {turn}')
    print()

## Part 4 — The LangGraph Workflow

Three-node pattern — same shape as the Redis workflow (example 82), different memory backend:

1. **`retrieve_memory`** — embed the query, fetch top-*k* relevant turns from ChromaDB
2. **`respond`** — answer using only the retrieved context (bounded, constant-size prompt!)
3. **`store_memory`** — embed and persist the new user turn and the assistant's response

The key insight: the context passed to the LLM is always `top_k` turns long — it never grows as history accumulates.

In [ ]:
from typing import TypedDict
from langchain_core.messages import HumanMessage
from langchain_openai import ChatOpenAI
from langgraph.graph import END, START, StateGraph

llm = ChatOpenAI(model='gpt-5.4-nano', temperature=0)

class VectorMemoryState(TypedDict):
    query: str
    relevant_history: list[str]
    answer: str

def retrieve_memory_node(state: VectorMemoryState) -> dict:
    relevant = retrieve_relevant(state['query'], top_k=3)
    print(f'  [retrieve] Found {len(relevant)} relevant turns')
    return {'relevant_history': relevant}

def respond_node(state: VectorMemoryState) -> dict:
    context = '\n'.join(state['relevant_history']) if state['relevant_history'] else '(no relevant history)'
    prompt = f'Relevant history:\n{context}\n\nUser: {state["query"]}\nAnswer concisely.'
    response = llm.invoke([HumanMessage(content=prompt)])
    return {'answer': response.content.strip()}

def store_memory_node(state: VectorMemoryState) -> dict:
    store_turn(state['query'], 'user')
    store_turn(state['answer'], 'assistant')
    return {}

builder = StateGraph(VectorMemoryState)
builder.add_node('retrieve_memory', retrieve_memory_node)
builder.add_node('respond',         respond_node)
builder.add_node('store_memory',    store_memory_node)
builder.add_edge(START,             'retrieve_memory')
builder.add_edge('retrieve_memory', 'respond')
builder.add_edge('respond',         'store_memory')
builder.add_edge('store_memory',    END)
app = builder.compile()

print('Workflow compiled. Running queries...\n')

for query in test_queries:
    print(f'Q: {query}')
    result = app.invoke({'query': query, 'relevant_history': [], 'answer': ''})
    print(f'A: {result["answer"]}\n')

## Part 5 — Scaling to Thousands of Turns

With **Redis**, loading 1 000 turns costs ~100 K tokens every request — and the cost grows with each new turn.

With **ChromaDB**, you always load `top_k=3` turns — **constant token cost** regardless of how many turns exist.

```
History size →  100    500    1000    5000
Redis context   10K    50K    100K    500K   (grows linearly)
Chroma context  ~300   ~300   ~300    ~300   (flat — always top_k)
```

The HNSW index in ChromaDB makes retrieval sub-linear (O(log N)) — even 100 000 turns retrieves in milliseconds.

In [ ]:
# Add 20 more 'noisy' turns about unrelated topics
noisy_turns = [
    ('I went hiking last weekend.', 'user'),
    ('That sounds relaxing!', 'assistant'),
    ('My coffee order is a flat white.', 'user'),
    ('Good choice — flat whites are smooth.', 'assistant'),
] * 5  # 20 turns total

for text, role in noisy_turns:
    store_turn(text, role)

total_turns = collection.count()
print(f'Total turns in Chroma: {total_turns}')
print()

# Query still retrieves only relevant turns — noisy turns are ignored
query = 'What ML framework do I prefer?'
relevant = retrieve_relevant(query, top_k=3)
print(f'Query: \'{query}\'')
print(f'Retrieved {len(relevant)} of {total_turns} turns:')
for turn in relevant:
    print(f'  - {turn}')
print('\nOnly relevant turns retrieved — noisy turns correctly ignored.')

## Exercises

**Exercise 1** — Change `top_k` from 3 to 1, then to 5. Run the same query. How does answer quality change with each setting? What is the trade-off between recall and token cost?

**Exercise 2** — Store turns for a second user (`SECOND_USER = "learner-99"`) and verify that queries for the original user (`learner-87`) do not retrieve the second user's turns. This exercises the `where={"user_id": ...}` filter.

In [ ]:
# Exercise 2 — User isolation
SECOND_USER = 'learner-99'

def store_turn_for_user(text: str, role: str, user_id: str) -> None:
    embedding = embeddings.embed_query(text)
    collection.add(
        ids=[str(uuid.uuid4())],
        embeddings=[embedding],
        documents=[text],
        metadatas=[{'user_id': user_id, 'role': role}],
    )

# Store turns for a different user
second_user_turns = [
    ('My name is Casey and I\'m a frontend developer.', 'user'),
    ('Nice to meet you, Casey!', 'assistant'),
    ('I use TypeScript and React every day.', 'user'),
]
for text, role in second_user_turns:
    store_turn_for_user(text, role, SECOND_USER)

# Query for original user — should NOT see Casey's turns
def retrieve_for_user(query: str, user_id: str, top_k: int = 3) -> list[str]:
    user_turns = collection.get(where={'user_id': user_id}, include=[])['ids']
    if not user_turns:
        return []
    query_embedding = embeddings.embed_query(query)
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=min(top_k, len(user_turns)),
        where={'user_id': user_id},
    )
    return results['documents'][0] if results['documents'] else []

print('Query for learner-87 (Jordan):')
for turn in retrieve_for_user('What\'s my name?', USER_ID, top_k=2):
    print(f'  - {turn}')

print('\nQuery for learner-99 (Casey):')
for turn in retrieve_for_user('What\'s my name?', SECOND_USER, top_k=2):
    print(f'  - {turn}')

print('\nUser isolation confirmed: each user sees only their own history.')

---

**Next:** [88 — Memory Architecture](../88-memory-architecture/memory_architecture_workbook.ipynb) — three-tier memory with episodic, semantic, and procedural tiers.